# 08 — HPO for the three selected-feature models

Notebook 07でモデル別に固定した特徴量を使い、XGBoost、Logistic Regression、MLPをOptunaで
最適化します。探索中に特徴量を選び直さないことで、「特徴量集合」と「ハイパーパラメータ」の
探索を混ぜません。

HPOはseed=2025のselection CVで行い、best parameterを確定してからseed=42の元foldで
confirmation OOF/test予測を一度だけ生成します。

In [ ]:
from functools import partial
from pathlib import Path
import sys

import optuna
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import Baseline
from features import get_base_features, load_nested_pair_te
from hpo import (
    HPO_MODELS,
    load_json,
    model_slug,
    save_json,
    suggest_model_params,
)
from train import make_model, run_cv

TARGET = Baseline.TARGET
ID_COLUMN = Baseline.ID_COLUMN
FOLD_COLUMN = Baseline.FOLD_COLUMN

train = pd.read_csv(ROOT / "input" / "train.csv")
test = pd.read_csv(ROOT / "input" / "test.csv")
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0})
assert train[TARGET].notna().all()

BASE_FEATURES, BASE_NUM, BASE_CAT = get_base_features(
    train, TARGET, ID_COLUMN, min_nunique=Baseline.MIN_NUMERIC_UNIQUE
)
selection = load_json(ROOT / "artifacts" / "selected_te_features.json")

selection_folds = pd.read_csv(ROOT / "output" / "selection_folds.csv")
selection_train = train.merge(
    selection_folds, on=ID_COLUMN, how="left", validate="one_to_one"
)
confirmation_folds = pd.read_csv(ROOT / "output" / "stkfolds.csv")
confirmation_train = train.merge(
    confirmation_folds, on=ID_COLUMN, how="left", validate="one_to_one"
)
assert selection_train[FOLD_COLUMN].notna().all()
assert confirmation_train[FOLD_COLUMN].notna().all()

SELECTION_TE_DIR = ROOT / "output" / "nested_pair_te_selection"
CONFIRMATION_TE_DIR = ROOT / "output" / "nested_pair_te"

## 探索空間と予算

- XGBoost: 木数、深さ、学習率、sampling、L1/L2
- Logistic Regression: `C`とclass weight
- MLP: 層構成、activation、L2、batch size、学習率

sampler seedとSQLite storageを固定します。同じNotebookを再実行してもstudyを再開できます。
`N_TRIALS`は追加trial数ではなく、完了trialの総数として扱います。

In [ ]:
N_TRIALS = {
    "XGBoost": 30,
    "LogisticRegression": 20,
    "MLP": 30,
}
BASELINE_TRIALS = {
    "XGBoost": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 1e-4,
        "reg_lambda": 1.0,
    },
    "LogisticRegression": {"C": 1.0, "class_weight": "none"},
    "MLP": {
        "architecture": "32_16",
        "activation": "relu",
        "alpha": 1e-3,
        "batch_size": 64,
        "learning_rate_init": 1e-3,
    },
}

## Optuna objective

各trialではselection foldのOOF AUCだけを返し、test予測やartifact保存は行いません。

In [ ]:
def make_objective(model_name, selected_te_features):
    numeric_features = BASE_NUM + selected_te_features
    if selected_te_features:
        loader = partial(
            load_nested_pair_te,
            feature_dir=SELECTION_TE_DIR,
            selected_te_features=selected_te_features,
        )
    else:
        loader = None

    def objective(trial):
        params = suggest_model_params(trial, model_name)
        trial.set_user_attr("resolved_model_params", params)
        model = make_model(
            model_name,
            numeric_features,
            BASE_CAT,
            seed=Baseline.SEED,
            use_gpu=False,
            model_params=params,
        )
        result = run_cv(
            model=model,
            train=selection_train,
            test=test,
            features=BASE_FEATURES,
            target=TARGET,
            id_column=ID_COLUMN,
            fold_column=FOLD_COLUMN,
            label=f"hpo_{model_name}_trial_{trial.number}",
            fold_feature_loader=loader,
            predict_test=False,
        )
        trial.set_user_attr(
            "fold_auc", result["fold_df"]["auc"].tolist()
        )
        return result["oof_auc"]

    return objective

In [ ]:
HPO_DIR = ROOT / "artifacts" / "hpo"
HPO_DIR.mkdir(parents=True, exist_ok=True)
best_payload = {}

for model_name in HPO_MODELS:
    selected_te = selection["models"][model_name]["selected_features"]
    slug = model_slug(model_name)
    storage_path = (HPO_DIR / f"{slug}.db").resolve().as_posix()
    study = optuna.create_study(
        study_name=f"customer_churn_{slug}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=Baseline.SEED),
        storage=f"sqlite:///{storage_path}",
        load_if_exists=True,
    )

    if len(study.trials) == 0:
        study.enqueue_trial(BASELINE_TRIALS[model_name])

    completed = sum(
        trial.state == optuna.trial.TrialState.COMPLETE
        for trial in study.trials
    )
    remaining = max(0, N_TRIALS[model_name] - completed)
    if remaining:
        study.optimize(
            make_objective(model_name, selected_te),
            n_trials=remaining,
            gc_after_trial=True,
        )

    best_params = study.best_trial.user_attrs["resolved_model_params"]
    best_payload[model_name] = {
        "selected_features": selected_te,
        "best_tuning_oof_auc": float(study.best_value),
        "best_trial_number": int(study.best_trial.number),
        "model_params": best_params,
    }
    study.trials_dataframe().to_csv(
        HPO_DIR / f"{slug}_trials.csv", index=False
    )

save_json(best_payload, HPO_DIR / "best_params.json")
display(
    pd.DataFrame(
        [
            {
                "model": name,
                "selected_features": len(result["selected_features"]),
                "best_tuning_oof_auc": result["best_tuning_oof_auc"],
                "best_trial_number": result["best_trial_number"],
            }
            for name, result in best_payload.items()
        ]
    )
)

## Frozen confirmation CV

ここから先は探索しません。モデルごとに選択済み列とbest parameterを固定し、seed=42の元foldで
OOF/test予測を作ります。Hill Climbingは次のNotebookでこの3候補だけを読み込みます。

In [ ]:
confirmation_rows = []
confirmation_results = {}

for model_name in HPO_MODELS:
    resolved = best_payload[model_name]
    selected_te = resolved["selected_features"]
    numeric_features = BASE_NUM + selected_te
    if selected_te:
        loader = partial(
            load_nested_pair_te,
            feature_dir=CONFIRMATION_TE_DIR,
            selected_te_features=selected_te,
        )
    else:
        loader = None

    model = make_model(
        model_name,
        numeric_features,
        BASE_CAT,
        seed=Baseline.SEED,
        use_gpu=False,
        model_params=resolved["model_params"],
    )
    prefix = f"final_{model_slug(model_name)}"
    result = run_cv(
        model=model,
        train=confirmation_train,
        test=test,
        features=BASE_FEATURES,
        target=TARGET,
        id_column=ID_COLUMN,
        fold_column=FOLD_COLUMN,
        label=f"final_{model_name}",
        save_prefix=prefix,
        output_dir=ROOT / "artifacts",
        fold_feature_loader=loader,
    )
    confirmation_results[model_name] = result
    confirmation_rows.append(
        {
            "model": model_name,
            "n_selected_te": len(selected_te),
            "tuning_oof_auc": resolved["best_tuning_oof_auc"],
            "confirmation_oof_auc": result["oof_auc"],
            "fold_auc_std": result["fold_df"]["auc"].std(ddof=0),
        }
    )

confirmation_summary = pd.DataFrame(confirmation_rows).sort_values(
    "confirmation_oof_auc", ascending=False
)
confirmation_summary.to_csv(
    ROOT / "artifacts" / "final_model_comparison.csv", index=False
)
display(confirmation_summary.reset_index(drop=True))

## HPO結果の読み方

tuning AUCよりconfirmation AUCが低くても、実装失敗とは限りません。trialを比較した分だけbest tuning
scoreにはwinner's curseが入ります。confirmationで元パラメータより改善しないモデルは、元モデルも
ensemble候補として残す判断ができます。この例では流れを明確にするため、次のNotebookへ渡す候補を
`final_*`の3モデルへ限定しています。